<a href="https://colab.research.google.com/github/tharak-bairneni/pandas-sales-data-analysis/blob/main/Pandas_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Load and Inspect the Data

In [1]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)

# Basic info
df.info()

# Missing values count
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB


,0
transaction_id,0
date,0
region,3
product_category,2
sales_amount,4
quantity,3
customer_age,4
payment_method,3


TASK 2 — Handle Missing Values

In [2]:
# Fill region & product_category with mode
df['region'].fillna(df['region'].mode()[0], inplace=True)
df['product_category'].fillna(df['product_category'].mode()[0], inplace=True)

# Fill sales_amount with median
df['sales_amount'].fillna(df['sales_amount'].median(), inplace=True)

# Forward fill quantity
df['quantity'].fillna(method='ffill', inplace=True)

# Fill customer_age with rounded mean
df['customer_age'].fillna(round(df['customer_age'].mean()), inplace=True)

# Drop rows where payment_method is missing
df.dropna(subset=['payment_method'], inplace=True)

# Verify no missing values
df.isna().sum()

/tmp/ipython-input-479/3362303425.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['region'].fillna(df['region'].mode()[0], inplace=True)
/tmp/ipython-input-479/3362303425.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpla

,0
transaction_id,0
date,0
region,0
product_category,0
sales_amount,0
quantity,0
customer_age,0
payment_method,0


TASK 3 — GroupBy Analysis

Total Sales by Region

In [5]:
#Calculate total sales by region

df.groupby('region')['sales_amount'].sum()

,sales_amount
region,
East,3790.0
North,6460.0
South,1900.0
West,2230.0


Average Sales by Product Category

In [6]:
#Calculate average sales by product_category

df.groupby('product_category')['sales_amount'].mean()

,sales_amount
product_category,
Books,508.333333
Clothing,680.000000
Electronics,1381.250000
Home,812.500000


Group by Region & Product Category

In [7]:
#Group by both region and product_category, calculate total sales and quantity, then use reset_index() to flatten

grouped = df.groupby(['region', 'product_category']).agg({
    'sales_amount': 'sum',
    'quantity': 'sum'
}).reset_index()

grouped

,region,product_category,sales_amount,quantity
0,East,Books,800.0,4.0
1,East,Clothing,890.0,3.0
2,East,Electronics,2100.0,1.0
3,North,Clothing,510.0,3.0
4,North,Electronics,2700.0,3.0
5,North,Home,3250.0,12.0
6,South,Clothing,1900.0,9.0
7,West,Books,725.0,1.0
8,West,Clothing,780.0,5.0
9,West,Electronics,725.0,1.0


Region-Product Combinations by Sales

In [8]:
#Display top 3 region-product combinations by sales
grouped.sort_values(by='sales_amount', ascending=False).head(3)

,region,product_category,sales_amount,quantity
5,North,Home,3250.0,12.0
4,North,Electronics,2700.0,3.0
2,East,Electronics,2100.0,1.0


TASK 4 — Custom Aggregation

Create Sales Range Function

In [13]:

def sales_range(series):
    return series.max() - series.min()

df.groupby('region')['sales_amount'].apply(sales_range)

,sales_amount
region,
East,1720.0
North,990.0
South,275.0
West,55.0


Multiple Aggregations Using .agg()

In [12]:

df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max'],
    'quantity': ['sum', 'min']
})

sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0